# Intraday Entry/Exit Strategy (TensorFlow)
This notebook provides a TensorFlow implementation of a dueling DQN LSTM policy (entry/exit) adapted to the `SuperTrader.AI` codebase.

Notes: the repository uses a lookback=30 and base features=32 (see `models/dqn_network.py`). This notebook runs a short smoke-test and demonstrates the policy + GPU setup.

## 1) Install & GPU setup
Install dependencies if missing and configure TensorFlow to use GPU. In hosted runtimes (Colab) switch runtime type to GPU. Locally, ensure CUDA/cuDNN and matching TensorFlow build are installed.

In [7]:
# Install common packages (uncomment to run)
# !pip install --upgrade pip
# !pip install tensorflow pandas numpy matplotlib scikit-learn joblib

import os
import tensorflow as tf
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('intraday-notebook')

# Print TF + GPU info
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs found:', gpus)
if gpus:
    try:
        # Enable memory growth for GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), 'Physical GPUs,', len(logical_gpus), 'Logical GPUs')
    except Exception as e:
        print('Could not set memory growth:', e)
else:
    print('No GPU available - the code will still run on CPU.')

TensorFlow version: 2.10.1
GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
1 Physical GPUs, 1 Logical GPUs


## 1) Install & GPU setup
Install dependencies if missing and configure TensorFlow to use GPU. In hosted runtimes (Colab) switch runtime type to GPU. Locally, ensure CUDA/cuDNN and matching TensorFlow build are installed.

In [8]:
# Install common packages (uncomment to run)
# !pip install --upgrade pip
# !pip install tensorflow pandas numpy matplotlib scikit-learn joblib

import os
import tensorflow as tf
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('intraday-notebook')

# Print TF + GPU info
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs found:', gpus)
if gpus:
    try:
        # Enable memory growth for GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), 'Physical GPUs,', len(logical_gpus), 'Logical GPUs')
    except Exception as e:
        print('Could not set memory growth:', e)
else:
    print('No GPU available - the code will still run on CPU.')

TensorFlow version: 2.10.1
GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
1 Physical GPUs, 1 Logical GPUs


## 2) Add codebase to PYTHONPATH and import models
Make the repo importable so we can reference `models.dqn_network` and the environment.

In [9]:
import sys
from pathlib import Path

# Fix the path (remove the extra backslash)
repo_root = Path('c:/Users/Jugaad/Documents/GitHub/SuperTrader.AI').resolve()
models_path = repo_root / 'models'

# Add both paths to sys.path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(models_path) not in sys.path:
    sys.path.insert(0, str(models_path))

print('Added to PYTHONPATH:')
print(f'  - {repo_root}')
print(f'  - {models_path}')

# Clear any cached imports
import importlib
for module_name in list(sys.modules.keys()):
    if 'dqn_network' in module_name or 'intraday_environment' in module_name or 'training' in module_name:
        del sys.modules[module_name]

# Try multiple import strategies
dqn_network = None
intraday_environment = None
training_module = None

print('\nTrying import strategies...')

# Strategy 1: Direct module imports
try:
    import dqn_network
    import intraday_environment
    import training as training_module
    print('✓ Strategy 1 SUCCESS: Direct imports worked')
except Exception as e1:
    print(f'✗ Strategy 1 failed: {e1}')
    
    # Strategy 2: Package imports
    try:
        import models.dqn_network as dqn_network
        import models.intraday_environment as intraday_environment
        import models.training as training_module
        print('✓ Strategy 2 SUCCESS: Package imports worked')
    except Exception as e2:
        print(f'✗ Strategy 2 failed: {e2}')
        
        # Strategy 3: importlib
        try:
            import importlib.util
            
            # Import dqn_network
            spec = importlib.util.spec_from_file_location("dqn_network", models_path / "dqn_network.py")
            dqn_network = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(dqn_network)
            
            # Import intraday_environment
            spec = importlib.util.spec_from_file_location("intraday_environment", models_path / "intraday_environment.py")
            intraday_environment = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(intraday_environment)
            
            # Import training
            spec = importlib.util.spec_from_file_location("training", models_path / "training.py")
            training_module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(training_module)
            
            print('✓ Strategy 3 SUCCESS: importlib worked')
        except Exception as e3:
            print(f'✗ Strategy 3 failed: {e3}')
            print('All import strategies failed - continuing with demo mode')

# Check what we got
if dqn_network is not None:
    print(f'✓ dqn_network imported: {type(dqn_network)}')
if intraday_environment is not None:
    print(f'✓ intraday_environment imported: {type(intraday_environment)}')
if training_module is not None:
    print(f'✓ training_module imported: {type(training_module)}')

Added to PYTHONPATH:
  - C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI
  - C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI\models

Trying import strategies...
✗ Strategy 1 failed: cannot import name 'IntradayMarket' from 'intraday_environment' (C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI\models\intraday_environment.py)
✗ Strategy 2 failed: cannot import name 'IntradayMarket' from 'intraday_environment' (C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI\models\intraday_environment.py)
✗ Strategy 3 failed: cannot import name 'IntradayMarket' from 'intraday_environment' (C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI\models\intraday_environment.py)
All import strategies failed - continuing with demo mode
✓ dqn_network imported: <class 'module'>
✓ intraday_environment imported: <class 'module'>
✓ training_module imported: <class 'module'>


## 3) Inspect model source files (quick programmatic view)
Print key model files so you can confirm class names and adapt calls.

In [10]:
files = [
    repo_root / 'models' / '__init__.py',
    repo_root / 'models' / 'dqn_network.py',
    repo_root / 'models' / 'intraday_environment.py',
    repo_root / 'models' / 'training.py'
]
for f in files:
    print('---', f.name, '---')
    try:
        print(f.read_text()[:4000])  # print first chunk
    except Exception as e:
        print('Could not read', f, 'Error:', e)

--- __init__.py ---
"""
SuperTrader.AI Models Package

This package contains the core trading models and environments for the SuperTrader.AI system.
"""

# Import main classes and functions to make them available at package level
from .dqn_network import *
from .intraday_environment import *
from .training import *

__all__ = [
    # DQN Network components
    'DQNNetwork',
    'DuelingDQN', 
    'PolicyNetwork',
    
    # Environment components  
    'IntradayTradingEnv',
    'TradingEnvironment',
    
    # Training components
    'DQNTrainer',
    'train_model',
    'evaluate_model',
]
--- dqn_network.py ---
"""
SIMPLIFIED DQN NETWORK - THE TRADING BRAIN

This is the AI "brain" that learns to make trading decisions.

WHAT IT DOES:
- Takes in market data (prices, indicators, time-of-day)
- Outputs 3 possible actions: BUY (long), SELL (short), or HOLD (stay flat)
- Learns from experience using "Deep Q-Learning"

ARCHITECTURE:
  Input (market state) 
    ↓
  LSTM Layer 1 (32 units) - 

## 4) Prepare dataset (placeholder + example feature engineering)
You should replace the CSV path with your actual intraday price data. Example shows common intraday features and scalers.

In [11]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Look for processed data in repo: prefer data/processed_data/train.csv then test.csv
processed_dir = repo_root / 'data' / 'processed_data'
df = None
if processed_dir.exists():
    candidates = ['train.csv', 'test.csv', 'test.csv']
    for fn in candidates:
        p = processed_dir / fn
        if p.exists():
            try:
                # Try to infer index column (common patterns: timestamp or index 0)
                df = pd.read_csv(p, parse_dates=['timestamp'], index_col='timestamp')
                print(f'Loaded processed file: {p} with', len(df), 'rows')
                break
            except Exception:
                try:
                    df = pd.read_csv(p, parse_dates=True, index_col=0)
                    print(f'Loaded processed file (index_col=0): {p} with', len(df), 'rows')
                    break
                except Exception as e:
                    print('Failed to read', p, 'Error:', e)
else:
    print('No processed_data folder found or no candidate files present; falling back to synthetic demo data')

# If no processed file found, create synthetic demo dataset (same as before)
if df is None:
    np.random.seed(42)
    timestamps = pd.date_range('2025-01-01 09:15', periods=360, freq='1min')
    base = 22000 + np.cumsum(np.random.randn(len(timestamps)) * 2)
    df = pd.DataFrame({
        'open': base + np.random.randn(len(timestamps)),
        'high': base + np.abs(np.random.randn(len(timestamps)) * 2),
        'low': base - np.abs(np.random.randn(len(timestamps)) * 2),
        'close': base,
        'volume': np.random.randint(100, 5000, len(timestamps))
    }, index=timestamps)
    print('Created synthetic data with', len(df), 'bars')

# Ensure all columns are numeric
df = df.select_dtypes(include=[np.number])
print('Data columns after numeric filtering:', df.columns.tolist())

# Basic feature engineering (works for both processed and synthetic inputs)
if 'close' in df.columns:
    df['ret_1'] = df['close'].pct_change().fillna(0)
    df['ret_5'] = df['close'].pct_change(5).fillna(0)
    df['sma_20'] = df['close'].rolling(20).mean().bfill().fillna(df['close'].iloc[0])
else:
    df['ret_1'] = 0
    df['ret_5'] = 0
    df['sma_20'] = 0

# Fix the vol_ewma calculation
if 'ret_1' in df.columns:
    df['vol_ewma'] = df['ret_1'].ewm(span=30).std().fillna(0.001)
else:
    df['vol_ewma'] = df.iloc[:, 0].pct_change().fillna(0).ewm(span=30).std().fillna(0.001)

# Choose feature columns adaptively (prefer common OHLCV + engineered features)
possible = ['open','high','low','close','volume','ret_1','ret_5','rsi_14','sma_20','vol_ewma']
feature_cols = [c for c in possible if c in df.columns]

# If rsi_14 missing, try to compute a simple proxy
if 'rsi_14' not in df.columns and 'ret_1' in df.columns:
    # Simple RSI calculation
    gains = df['ret_1'].where(df['ret_1'] > 0, 0)
    losses = -df['ret_1'].where(df['ret_1'] < 0, 0)
    avg_gains = gains.rolling(14).mean()
    avg_losses = losses.rolling(14).mean()
    rs = avg_gains / (avg_losses + 1e-10)  # Add small value to avoid division by zero
    df['rsi_14'] = 100 - (100 / (1 + rs))
    df['rsi_14'] = df['rsi_14'].fillna(50)  # Fill NaN with neutral RSI value
    if 'rsi_14' not in feature_cols:
        feature_cols.append('rsi_14')

# If feature_cols is still too small, take first 10 numeric columns
if len(feature_cols) < 6:
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = numeric[:min(10, len(numeric))]

# Ensure all feature columns contain only numeric data
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

print('Selected feature columns:', feature_cols)
print('Feature data types:', df[feature_cols].dtypes.to_dict())

# Scale the features
scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols])
print('Feature matrix shape:', X.shape, 'Using feature_cols:', feature_cols)

Loaded processed file (index_col=0): C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI\data\processed_data\train.csv with 7590 rows
Data columns after numeric filtering: ['open', 'high', 'low', 'close', 'volume', 'norm_close', 'returns', 'log_returns', 'rsi_30', 'macd', 'atr_14', 'ema_9', 'ema_21', 'ema_distance', 'volatility', 'hour', 'minute', 'minutes_since_open', 'minutes_to_close', 'session']
Selected feature columns: ['open', 'high', 'low', 'close', 'volume', 'ret_1', 'ret_5', 'sma_20', 'vol_ewma', 'rsi_14']
Feature data types: {'open': dtype('float64'), 'high': dtype('float64'), 'low': dtype('float64'), 'close': dtype('float64'), 'volume': dtype('int64'), 'ret_1': dtype('float64'), 'ret_5': dtype('float64'), 'sma_20': dtype('float64'), 'vol_ewma': dtype('float64'), 'rsi_14': dtype('float64')}
Feature matrix shape: (7590, 10) Using feature_cols: ['open', 'high', 'low', 'close', 'volume', 'ret_1', 'ret_5', 'sma_20', 'vol_ewma', 'rsi_14']


## 5) Instantiate IntradayEnvironment (example)
The repository environment class may be named `IntradayTradingEnv` (or similar). We'll attempt to instantiate it if available, otherwise we use a tiny wrapper to simulate step/reset semantics.

In [12]:
# Clear any cached imports first
import importlib
import sys

# Clear any cached modules
for module_name in list(sys.modules.keys()):
    if 'intraday_environment' in module_name:
        del sys.modules[module_name]
        print(f"Cleared cached module: {module_name}")

# Try to instantiate environment from repository if present
EnvClass = None
env = None

try:
    # Reimport the clean module
    import models.intraday_environment as intraday_environment
    importlib.reload(intraday_environment)
    
    # First, let's check what's actually available in the intraday_environment module
    if intraday_environment is not None:
        print("Available classes/functions in intraday_environment:")
        for attr in dir(intraday_environment):
            if not attr.startswith('_'):
                print(f"  - {attr}: {type(getattr(intraday_environment, attr))}")
        
        # Try different possible class names
        possible_classes = ['IntradayTradingEnv', 'IntradayMarket', 'TradingEnvironment', 'DuelingDQN_LSTM']
        for class_name in possible_classes:
            if hasattr(intraday_environment, class_name):
                EnvClass = getattr(intraday_environment, class_name)
                print(f"Found class: {class_name}")
                break
        
        # If we found IntradayTradingEnv, try to instantiate it
        if EnvClass is not None and 'IntradayTradingEnv' in str(EnvClass):
            # Prepare data in the expected format for IntradayTradingEnv
            # Add required columns if missing
            df_copy = df.copy()
            
            # Add date column if missing (required by IntradayTradingEnv)
            if 'date' not in df_copy.columns:
                df_copy['date'] = pd.to_datetime(df_copy.index).date
            
            # Ensure we have all required OHLCV columns
            required_cols = ['open', 'high', 'low', 'close', 'volume']
            for col in required_cols:
                if col not in df_copy.columns:
                    if col == 'volume':
                        df_copy[col] = 1000  # Default volume
                    else:
                        df_copy[col] = df_copy.get('close', df_copy.iloc[:, 0])  # Use close or first column
            
            try:
                env = EnvClass(
                    data=df_copy,
                    initial_capital=500000,
                    lot_size=50,
                    max_lots=3,
                    target_volatility=0.12,  # 12% for intraday
                    transaction_cost_bp=10,   # 10 basis points
                    stt_sell_pct=0.025       # 0.025% STT
                )
                print(f'✓ Successfully instantiated {class_name}')
                print(f'  - Trading days: {len(env.trading_days) if hasattr(env, "trading_days") else "N/A"}')
                print(f'  - Feature count: {env._get_feature_count() if hasattr(env, "_get_feature_count") else "N/A"}')
                
                # *** CRITICAL FIX: Update feature_cols to match environment ***
                test_state = env.reset()
                env_feature_count = test_state.shape[1]
                print(f'  - Environment state features: {env_feature_count}')
                
                # Get environment's actual feature columns if available
                if hasattr(env, 'feature_cols'):
                    feature_cols = env.feature_cols
                    print(f'  - Environment feature columns: {feature_cols}')
                elif hasattr(env, '_get_feature_names'):
                    feature_cols = env._get_feature_names()
                    print(f'  - Environment feature names: {feature_cols}')
                else:
                    # Fallback: use first N columns from our processed data
                    all_cols = df_copy.select_dtypes(include=[np.number]).columns.tolist()
                    feature_cols = all_cols[:env_feature_count]
                    print(f'  - Using fallback feature columns: {feature_cols}')
                
                print(f'✓ Updated feature_cols to match environment: {len(feature_cols)} features')
                
            except Exception as init_error:
                print(f'Failed to initialize {class_name}: {init_error}')
                env = None
        else:
            print("No suitable environment class found")
            
    if env is None:
        raise ImportError('Could not instantiate repository environment')
        
except Exception as e:
    print(f'Could not instantiate repo environment (falling back to demo wrapper). Error: {e}')
    
    # Enhanced Demo fallback environment with better compatibility
    class DemoEnv:
        def __init__(self, df, window=30):
            self.df = df.copy()
            self.window = window
            self.step_idx = 0
            self.position = 0
            self.entry_price = 0.0
            self.initial_capital = 500000
            self.lot_size = 50
            
            # Ensure we have required columns
            if 'close' not in self.df.columns:
                # Use the first numeric column as close price
                numeric_cols = self.df.select_dtypes(include=[np.number]).columns
                if len(numeric_cols) > 0:
                    self.df['close'] = self.df[numeric_cols[0]]
                else:
                    self.df['close'] = 22000  # Default price
            
            print(f'DemoEnv initialized with {len(self.df)} bars')
            
        def reset(self):
            self.step_idx = self.window
            self.position = 0
            self.entry_price = 0.0
            return self._get_state()
            
        def _get_state(self):
            if self.step_idx < self.window:
                # Pad with zeros if we don't have enough history
                available_data = self.df.iloc[:self.step_idx][feature_cols].values
                padding = np.zeros((self.window - len(available_data), len(feature_cols)))
                state = np.vstack([padding, available_data]) if len(available_data) > 0 else np.zeros((self.window, len(feature_cols)))
            else:
                window_data = self.df.iloc[self.step_idx-self.window:self.step_idx]
                state = window_data[feature_cols].values
            
            return state.astype(np.float32)
            
        def step(self, action):
            # action: 0=short, 1=hold, 2=long (matching repo convention)
            if self.step_idx >= len(self.df):
                return self._get_state(), 0.0, True, {'reason': 'end_of_data'}
                
            price = float(self.df['close'].iloc[self.step_idx])
            reward = 0.0
            done = False
            info = {'position': self.position, 'price': price}
            
            # Position management (simplified)
            target_position = {0: -1, 1: 0, 2: 1}[action]  # Short, Hold, Long
            
            if target_position != self.position:
                # Position change - calculate P&L if exiting
                if self.position != 0:
                    pnl = (price - self.entry_price) * self.position * self.lot_size
                    reward = pnl / self.initial_capital  # Normalize by capital
                    info['trade_pnl'] = pnl
                
                # Update position
                old_position = self.position
                self.position = target_position
                
                # Set entry price for new position
                if self.position != 0:
                    self.entry_price = price
                else:
                    self.entry_price = 0.0
                    
                info['position_change'] = f"{old_position} -> {self.position}"
            
            # Move to next step
            self.step_idx += 1
            
            # Check if done
            if self.step_idx >= len(self.df):
                done = True
                # Force exit at end
                if self.position != 0:
                    final_pnl = (price - self.entry_price) * self.position * self.lot_size
                    reward += final_pnl / self.initial_capital
                    info['final_exit_pnl'] = final_pnl
            
            next_state = self._get_state() if not done else np.zeros((self.window, len(feature_cols)), dtype=np.float32)
            return next_state, float(reward), done, info
    
    env = DemoEnv(df)
    print('✓ Using enhanced DemoEnv for the smoke-run')
    print(f'  - Data points: {len(df)}')
    print(f'  - Feature columns: {feature_cols}')
    print(f'  - Window size: 30')

# Test the environment
print('\n--- Environment Test ---')
test_state = env.reset()
print(f'Initial state shape: {test_state.shape}')
test_action = 2  # Long
next_state, reward, done, info = env.step(test_action)
print(f'Step result - Reward: {reward:.6f}, Done: {done}, Info: {info}')
print(f'Next state shape: {next_state.shape}')

# FINAL VERIFICATION: Ensure feature_cols matches environment output
final_state = env.reset()
feature_cols_count = len(feature_cols)
env_state_count = final_state.shape[1]
if feature_cols_count != env_state_count:
    print(f'\n⚠️  WARNING: feature_cols count ({feature_cols_count}) != environment state count ({env_state_count})')
    print('This will cause errors in the model. Adjusting feature_cols...')
    # Take the first N feature names we have to match environment
    all_available = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = all_available[:env_state_count]
    print(f'✓ Adjusted feature_cols to: {feature_cols}')
else:
    print(f'✓ feature_cols count ({feature_cols_count}) matches environment state count ({env_state_count})')

Cleared cached module: intraday_environment
Cleared cached module: models.intraday_environment
Could not instantiate repo environment (falling back to demo wrapper). Error: cannot import name 'IntradayMarket' from 'intraday_environment' (C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI\models\intraday_environment.py)
DemoEnv initialized with 7590 bars
✓ Using enhanced DemoEnv for the smoke-run
  - Data points: 7590
  - Feature columns: ['open', 'high', 'low', 'close', 'volume', 'ret_1', 'ret_5', 'sma_20', 'vol_ewma', 'rsi_14']
  - Window size: 30

--- Environment Test ---
Initial state shape: (30, 10)
Step result - Reward: 0.000000, Done: False, Info: {'position': 0, 'price': 908.55, 'position_change': '0 -> 1'}
Next state shape: (30, 10)
✓ feature_cols count (10) matches environment state count (10)


## 6) Build / load DQN networks (policy & target)
We'll implement a TensorFlow Keras dueling LSTM matching the repo's shape: lookback=30, features ~ len(feature_cols) (demo).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers

class DuelingDQN_LSTM(Model):
    def __init__(self, n_features, n_actions=3, lstm1_units=64, lstm2_units=32, dropout=0.1):
        super().__init__()
        self.n_features = n_features  # Store for reference
        self.lstm1 = layers.LSTM(lstm1_units, return_sequences=True)
        self.lstm2 = layers.LSTM(lstm2_units, return_sequences=False)
        self.drop = layers.Dropout(dropout)
        self.val_fc = layers.Dense(lstm2_units, activation='relu')
        self.val_out = layers.Dense(1)
        self.adv_fc = layers.Dense(lstm2_units, activation='relu')
        self.adv_out = layers.Dense(n_actions)
    def call(self, x, training=False):
        x = self.lstm1(x, training=training)
        x = self.lstm2(x, training=training)
        x = self.drop(x, training=training)
        V = self.val_out(self.val_fc(x))
        A = self.adv_out(self.adv_fc(x))
        A_mean = tf.reduce_mean(A, axis=1, keepdims=True)
        Q = V + (A - A_mean)
        return Q

# Instantiate policy & target models for demo
n_features = len(feature_cols)
print(f'Building models with {n_features} features for columns: {feature_cols}')
policy_model = DuelingDQN_LSTM(n_features=n_features, n_actions=3)
target_model = DuelingDQN_LSTM(n_features=n_features, n_actions=3)

# Build models by calling with correct feature dimensions
dummy = tf.zeros((1, 30, n_features))  # Use actual n_features, not hardcoded value
print(f'Building with dummy tensor shape: {dummy.shape}')
policy_model(dummy)
target_model(dummy)

# Copy weights to target
target_model.set_weights(policy_model.get_weights())
optimizer = optimizers.Adam(learning_rate=1e-4)
print('Policy & target models created. Policy summary:')
policy_model.summary()

## 7) Entry & exit policy (epsilon-greedy + mapping)
Action mapping used in the repo: 0=SHORT, 1=HOLD, 2=LONG. We'll implement epsilon-greedy and enforce the time-safety rule: if minutes_to_close < 15 => HOLD.

In [ ]:
import numpy as np

class TFTradingAgent:
    def __init__(self, policy_model, target_model, n_actions=3, gamma=0.3, epsilon=0.5, epsilon_min=0.01, epsilon_decay=0.995):
        self.policy_model = policy_model
        self.target_model = target_model
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.optimizer = optimizers.Adam(learning_rate=1e-4)

    def decide_action(self, state, mode='train', minutes_to_close=180.0):
        # Force HOLD close to market close (safety rule from repo)
        if minutes_to_close < 15:
            return 1, {'reason': 'time_constraint'}

        if mode == 'train' and np.random.rand() < self.epsilon:
            return np.random.randint(0, self.n_actions), {'mode': 'explore', 'epsilon': self.epsilon}

        q = self.policy_model(state[np.newaxis].astype(np.float32))
        q = q.numpy()[0]
        action = int(np.argmax(q))
        return action, {'mode': 'exploit', 'q_values': q.tolist()}

    def remember(self, *args, **kwargs):
        # For demo, memory handled externally by ReplayBuffer below
        pass

    def train_on_batch(self, states, actions, rewards, next_states, dones):
        # Compute targets using Bellman equation with target_model
        next_q = self.target_model(next_states).numpy()
        max_next_q = np.max(next_q, axis=1)
        targets = rewards + (1 - dones) * self.gamma * max_next_q

        with tf.GradientTape() as tape:
            q_values = self.policy_model(states)
            # gather predicted q for actions
            action_masks = tf.one_hot(actions, self.n_actions)
            pred_q = tf.reduce_sum(q_values * action_masks, axis=1)
            loss = tf.reduce_mean(tf.keras.losses.Huber()(targets, pred_q))
        grads = tape.gradient(loss, self.policy_model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.policy_model.trainable_variables))

        # NOTE: epsilon decay moved out of train_on_batch so decay happens per-episode
        return float(loss)

    def decay_epsilon(self):
        """Decay exploration rate once per episode (or when explicitly called)."""
        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


## 8) Replay buffer & training utilities

In [ ]:
import collections, random
class ReplayBuffer:
    def __init__(self, capacity=5000):
        self.buf = collections.deque(maxlen=capacity)
    def add(self, s,a,r,ns,d):
        self.buf.append((s,a,r,ns,d))
    def sample(self, batch_size):
        batch = random.sample(self.buf, min(batch_size, len(self.buf)))
        s,a,r,ns,d = map(np.array, zip(*batch))
        return s, a, r.astype(np.float32), ns, d.astype(np.float32)
    def __len__(self):
        return len(self.buf)

buffer = ReplayBuffer(capacity=2000)

# Create agent with the correctly built models (after model creation cell)
agent = TFTradingAgent(policy_model, target_model)
print('ReplayBuffer & agent ready. Buffer size:', len(buffer))
print(f'Agent using models with feature dimensions: {policy_model.n_features}')

## 9) Training loop (smoke run for 1-2 episodes)
We'll run a short loop to demonstrate end-to-end action selection and training steps. This is NOT full training; adapt the loops and logging from `models/training.py` for full training.

In [ ]:
# Simple smoke training loop over the demo environment
episodes = 200
batch_size = 16
target_update_freq = 50
for ep in range(episodes):
    state = env.reset()
    done = False
    total_reward = 0.0
    step = 0
    while not done and step < 200:
        # For demo, minutes_to_close reduces with step (simple proxy)
        minutes_to_close = max(0, 180 - step)
        action, info = agent.decide_action(state, mode='train', minutes_to_close=minutes_to_close)
        next_state, reward, done, env_info = env.step(action)
        buffer.add(state, action, reward, next_state, float(done))
        state = next_state
        total_reward += reward
        step += 1
        # Train if buffer has enough samples
        if len(buffer) >= batch_size:
            s_b, a_b, r_b, ns_b, d_b = buffer.sample(batch_size)
            loss = agent.train_on_batch(s_b, a_b, r_b, ns_b, d_b)
        # Periodically sync target network
        if step % target_update_freq == 0:
            target_model.set_weights(policy_model.get_weights())
    print(f'Episode {ep+1} finished. Steps: {step}, Total reward: {total_reward:.4f}, Epsilon: {agent.epsilon:.3f}')

    # Decay epsilon once per episode (prevent rapid decay due to many train steps)
    agent.decay_epsilon()


## 9.1) Enhanced Training with Detailed Debugging
Let's create an improved training loop with comprehensive debugging to identify and fix the training issues.

In [ ]:
# Enhanced training loop with comprehensive debugging
import matplotlib.pyplot as plt
from collections import defaultdict
import time

# Reset the environment and agent for fresh training
print("=== DIAGNOSTIC SETUP ===")

# Create fresh agent with better parameters
class EnhancedTFTradingAgent(TFTradingAgent):
    def __init__(self, policy_model, target_model, n_actions=3, gamma=0.95, epsilon=0.9, epsilon_min=0.05, epsilon_decay=0.996):
        super().__init__(policy_model, target_model, n_actions, gamma, epsilon, epsilon_min, epsilon_decay)
        self.action_counts = defaultdict(int)
        self.q_value_history = []
        self.loss_history = []
    
    def decide_action(self, state, mode='train', minutes_to_close=180.0):
        action, info = super().decide_action(state, mode, minutes_to_close)
        self.action_counts[action] += 1
        
        # Track Q-values
        if mode == 'train':
            q_vals = self.policy_model(state[np.newaxis].astype(np.float32)).numpy()[0]
            self.q_value_history.append({
                'q_short': q_vals[0], 
                'q_hold': q_vals[1], 
                'q_long': q_vals[2],
                'selected_action': action,
                'epsilon': self.epsilon
            })
        
        return action, info
    
    def train_on_batch(self, states, actions, rewards, next_states, dones):
        loss = super().train_on_batch(states, actions, rewards, next_states, dones)
        self.loss_history.append(loss)
        return loss

# Create enhanced environment with better reward structure
class DiagnosticDemoEnv(env.__class__):
    def __init__(self, df, window=30):
        super().__init__(df, window)
        self.episode_stats = []
        self.current_episode = {'actions': [], 'rewards': [], 'positions': [], 'prices': []}
        
    def reset(self):
        # Save previous episode stats
        if self.current_episode['actions']:
            self.episode_stats.append(self.current_episode.copy())
        
        # Reset for new episode
        self.current_episode = {'actions': [], 'rewards': [], 'positions': [], 'prices': []}
        return super().reset()
    
    def step(self, action):
        next_state, reward, done, info = super().step(action)
        
        # Enhanced reward calculation
        price = float(self.df['close'].iloc[min(self.step_idx-1, len(self.df)-1)])
        
        # Track episode data
        self.current_episode['actions'].append(action)
        self.current_episode['rewards'].append(reward)
        self.current_episode['positions'].append(self.position)
        self.current_episode['prices'].append(price)
        
        # Improved reward shaping
        shaped_reward = reward
        
        # Add small positive reward for holding when market is stable
        if action == 1 and abs(reward) < 0.001:  # Hold with small/no loss
            shaped_reward += 0.0001
        
        # Penalty for excessive position changes
        if len(self.current_episode['actions']) > 1:
            prev_action = self.current_episode['actions'][-2]
            if action != prev_action and action != 1:  # Changed to non-hold
                shaped_reward -= 0.0005
        
        return next_state, shaped_reward, done, info

# Create diagnostic environment
print("Creating diagnostic environment...")
diag_env = DiagnosticDemoEnv(df)

# Test diagnostic environment
test_state = diag_env.reset()
print(f"Environment state shape: {test_state.shape}")
print(f"Data length: {len(diag_env.df)}")
print(f"Window size: {diag_env.window}")

# Create enhanced agent with better hyperparameters
print("Creating enhanced agent...")
enhanced_agent = EnhancedTFTradingAgent(policy_model, target_model, 
                                      gamma=0.95,      # Higher gamma for longer-term rewards
                                      epsilon=0.9,     # Start with more exploration
                                      epsilon_min=0.05, # Don't go too low
                                      epsilon_decay=0.996) # Slower decay

# Create larger buffer
enhanced_buffer = ReplayBuffer(capacity=10000)

print("=== DIAGNOSTIC TRAINING PARAMETERS ===")
print(f"Episodes: 50")
print(f"Batch size: 32")
print(f"Target update frequency: 25")
print(f"Initial epsilon: {enhanced_agent.epsilon}")
print(f"Gamma: {enhanced_agent.gamma}")

# Training tracking
training_stats = {
    'episode_rewards': [],
    'episode_steps': [],
    'epsilon_values': [],
    'losses': [],
    'action_distributions': []
}

print("\n=== STARTING ENHANCED TRAINING ===")
start_time = time.time()

episodes = 200
batch_size = 32
target_update_freq = 25

for ep in range(episodes):
    state = diag_env.reset()
    done = False
    total_reward = 0.0
    step = 0
    episode_actions = defaultdict(int)
    
    while not done and step < 200:  # Increased step limit
        # Gradual time constraint (more realistic)
        minutes_to_close = max(0, 375 - step * 2)  # 375 minutes = 6.25 hours
        
        action, info = enhanced_agent.decide_action(state, mode='train', 
                                                  minutes_to_close=minutes_to_close)
        episode_actions[action] += 1
        
        next_state, reward, done, env_info = diag_env.step(action)
        enhanced_buffer.add(state, action, reward, next_state, float(done))
        
        state = next_state
        total_reward += reward
        step += 1
        
        # Train more frequently with larger buffer
        if len(enhanced_buffer) >= batch_size and step % 2 == 0:
            s_b, a_b, r_b, ns_b, d_b = enhanced_buffer.sample(batch_size)
            loss = enhanced_agent.train_on_batch(s_b, a_b, r_b, ns_b, d_b)
        
        # Update target network less frequently
        if step % target_update_freq == 0:
            target_model.set_weights(policy_model.get_weights())
    
    # Track episode statistics
    training_stats['episode_rewards'].append(total_reward)
    training_stats['episode_steps'].append(step)
    training_stats['epsilon_values'].append(enhanced_agent.epsilon)
    training_stats['action_distributions'].append(dict(episode_actions))

    # Decay epsilon once per episode (prevent rapid decay due to many train steps)
    enhanced_agent.decay_epsilon()
    
    # Print detailed progress every 10 episodes
    if (ep + 1) % 10 == 0 or ep == 0:
        avg_reward = np.mean(training_stats['episode_rewards'][-10:])
        avg_steps = np.mean(training_stats['episode_steps'][-10:])
        recent_losses = enhanced_agent.loss_history[-10:] if enhanced_agent.loss_history else [0]
        avg_loss = np.mean(recent_losses)
        
        print(f"Episode {ep+1:3d} | "
              f"Reward: {total_reward:8.4f} | "
              f"Steps: {step:3d} | "
              f"Epsilon: {enhanced_agent.epsilon:.3f} | "
              f"Avg10 Reward: {avg_reward:8.4f} | "
              f"Avg Loss: {avg_loss:.4f}")
        
        print(f"    Actions - Short: {episode_actions[0]:2d}, "
              f"Hold: {episode_actions[1]:2d}, "
              f"Long: {episode_actions[2]:2d}")

elapsed_time = time.time() - start_time
print(f"\n=== TRAINING COMPLETED ===")
print(f"Training time: {elapsed_time:.2f} seconds")
print(f"Buffer size: {len(enhanced_buffer)}")
print(f"Total Q-value samples: {len(enhanced_agent.q_value_history)}")

# Analyze training results
print("\n=== TRAINING ANALYSIS ===")
print(f"Total episodes: {len(training_stats['episode_rewards'])}")
print(f"Average reward: {np.mean(training_stats['episode_rewards']):.4f}")
print(f"Best episode reward: {np.max(training_stats['episode_rewards']):.4f}")
print(f"Worst episode reward: {np.min(training_stats['episode_rewards']):.4f}")
print(f"Average steps per episode: {np.mean(training_stats['episode_steps']):.1f}")
print(f"Final epsilon: {enhanced_agent.epsilon:.4f}")

# Action distribution analysis
total_actions = enhanced_agent.action_counts
total_count = sum(total_actions.values())
print(f"\nOverall action distribution:")
print(f"  Short (0): {total_actions[0]:4d} ({100*total_actions[0]/total_count:.1f}%)")
print(f"  Hold  (1): {total_actions[1]:4d} ({100*total_actions[1]/total_count:.1f}%)")
print(f"  Long  (2): {total_actions[2]:4d} ({100*total_actions[2]/total_count:.1f}%)")

In [ ]:
# Comprehensive training analysis and visualization
import matplotlib.pyplot as plt
plt.style.use('default')
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('DQN Training Analysis', fontsize=16)

# 1. Episode rewards over time
axes[0,0].plot(training_stats['episode_rewards'], 'b-', alpha=0.7, linewidth=1)
axes[0,0].plot(np.convolve(training_stats['episode_rewards'], np.ones(5)/5, mode='valid'), 'r-', linewidth=2, label='5-episode moving average')
axes[0,0].set_title('Episode Rewards')
axes[0,0].set_xlabel('Episode')
axes[0,0].set_ylabel('Total Reward')
axes[0,0].grid(True, alpha=0.3)
axes[0,0].legend()

# 2. Episode steps over time
axes[0,1].plot(training_stats['episode_steps'], 'g-', linewidth=1)
axes[0,1].axhline(y=np.mean(training_stats['episode_steps']), color='r', linestyle='--', label=f'Mean: {np.mean(training_stats["episode_steps"]):.1f}')
axes[0,1].set_title('Episode Steps')
axes[0,1].set_xlabel('Episode')
axes[0,1].set_ylabel('Steps per Episode')
axes[0,1].grid(True, alpha=0.3)
axes[0,1].legend()

# 3. Epsilon decay
axes[0,2].plot(training_stats['epsilon_values'], 'purple', linewidth=2)
axes[0,2].set_title('Epsilon Decay (Exploration Rate)')
axes[0,2].set_xlabel('Episode')
axes[0,2].set_ylabel('Epsilon')
axes[0,2].grid(True, alpha=0.3)

# 4. Action distribution over episodes
action_data = {0: [], 1: [], 2: []}
for dist in training_stats['action_distributions']:
    total = sum(dist.values()) or 1  # Avoid division by zero
    action_data[0].append(dist.get(0, 0) / total * 100)
    action_data[1].append(dist.get(1, 0) / total * 100)
    action_data[2].append(dist.get(2, 0) / total * 100)

episodes_range = range(len(action_data[0]))
axes[1,0].plot(episodes_range, action_data[0], 'r-', label='Short (0)', linewidth=2)
axes[1,0].plot(episodes_range, action_data[1], 'b-', label='Hold (1)', linewidth=2)
axes[1,0].plot(episodes_range, action_data[2], 'g-', label='Long (2)', linewidth=2)
axes[1,0].set_title('Action Distribution Over Episodes (%)')
axes[1,0].set_xlabel('Episode')
axes[1,0].set_ylabel('Action Percentage')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 5. Loss over training steps
if enhanced_agent.loss_history:
    axes[1,1].plot(enhanced_agent.loss_history, 'orange', alpha=0.7, linewidth=1)
    if len(enhanced_agent.loss_history) > 10:
        smooth_loss = np.convolve(enhanced_agent.loss_history, np.ones(10)/10, mode='valid')
        axes[1,1].plot(range(9, len(enhanced_agent.loss_history)), smooth_loss, 'red', linewidth=2, label='10-step moving average')
    axes[1,1].set_title('Training Loss')
    axes[1,1].set_xlabel('Training Step')
    axes[1,1].set_ylabel('Huber Loss')
    axes[1,1].grid(True, alpha=0.3)
    axes[1,1].legend()
else:
    axes[1,1].text(0.5, 0.5, 'No loss data available', ha='center', va='center', transform=axes[1,1].transAxes)
    axes[1,1].set_title('Training Loss')

# 6. Q-value evolution (last 500 samples)
if enhanced_agent.q_value_history:
    recent_q = enhanced_agent.q_value_history[-500:]  # Last 500 Q-value samples
    q_short = [q['q_short'] for q in recent_q]
    q_hold = [q['q_hold'] for q in recent_q]
    q_long = [q['q_long'] for q in recent_q]
    
    axes[1,2].plot(q_short, 'r-', alpha=0.7, label='Q(Short)', linewidth=1)
    axes[1,2].plot(q_hold, 'b-', alpha=0.7, label='Q(Hold)', linewidth=1)
    axes[1,2].plot(q_long, 'g-', alpha=0.7, label='Q(Long)', linewidth=1)
    axes[1,2].set_title('Q-Values Evolution (Recent 500 samples)')
    axes[1,2].set_xlabel('Sample')
    axes[1,2].set_ylabel('Q-Value')
    axes[1,2].legend()
    axes[1,2].grid(True, alpha=0.3)
else:
    axes[1,2].text(0.5, 0.5, 'No Q-value data available', ha='center', va='center', transform=axes[1,2].transAxes)
    axes[1,2].set_title('Q-Values Evolution')

plt.tight_layout()
plt.show()

# Detailed diagnostic summary
print("\n" + "="*80)
print("DETAILED DIAGNOSTIC SUMMARY")
print("="*80)

print(f"\n📊 TRAINING METRICS:")
print(f"   Episodes completed: {len(training_stats['episode_rewards'])}")
print(f"   Total training steps: {sum(training_stats['episode_steps'])}")
print(f"   Average episode length: {np.mean(training_stats['episode_steps']):.1f} ± {np.std(training_stats['episode_steps']):.1f}")
print(f"   Experience buffer size: {len(enhanced_buffer)} / {enhanced_buffer.buf.maxlen}")

print(f"\n💰 REWARD ANALYSIS:")
rewards = training_stats['episode_rewards']
print(f"   Mean reward: {np.mean(rewards):8.4f}")
print(f"   Std deviation: {np.std(rewards):8.4f}")
print(f"   Median reward: {np.median(rewards):8.4f}")
print(f"   Min/Max reward: {np.min(rewards):8.4f} / {np.max(rewards):8.4f}")
print(f"   Positive episodes: {sum(1 for r in rewards if r > 0):3d} / {len(rewards)} ({100*sum(1 for r in rewards if r > 0)/len(rewards):.1f}%)")

print(f"\n🎯 ACTION ANALYSIS:")
total_actions = enhanced_agent.action_counts
total_count = sum(total_actions.values()) or 1
for action in [0, 1, 2]:
    action_name = ['Short', 'Hold', 'Long'][action]
    count = total_actions[action]
    percentage = 100 * count / total_count
    print(f"   {action_name:5s} ({action}): {count:6d} actions ({percentage:5.1f}%)")

print(f"\n🧠 LEARNING ANALYSIS:")
print(f"   Final epsilon: {enhanced_agent.epsilon:.4f}")
print(f"   Training batches: {len(enhanced_agent.loss_history)}")
if enhanced_agent.loss_history:
    recent_loss = enhanced_agent.loss_history[-10:]
    print(f"   Recent avg loss: {np.mean(recent_loss):.6f}")
    print(f"   Loss trend: {'Decreasing' if len(enhanced_agent.loss_history) > 20 and np.mean(enhanced_agent.loss_history[-10:]) < np.mean(enhanced_agent.loss_history[-20:-10]) else 'Stable/Increasing'}")

print(f"\n🔍 POTENTIAL ISSUES:")
issues = []
if np.mean(rewards) < -0.01:
    issues.append("Very negative average rewards - reward shaping may need adjustment")
if np.std(training_stats['episode_steps']) < 2:
    issues.append("Consistent episode lengths - environment may be terminating too early")
if total_actions[1] / total_count > 0.7:
    issues.append("Excessive holding - agent may be risk-averse or not learning entry/exit timing")
if enhanced_agent.epsilon < 0.1 and len(rewards) < 100:
    issues.append("Exploration ended too early - epsilon decay may be too aggressive")

if issues:
    for i, issue in enumerate(issues, 1):
        print(f"   {i}. {issue}")
else:
    print("   No obvious issues detected!")

print(f"\n💡 RECOMMENDATIONS:")
if np.mean(rewards) < -0.01:
    print("   - Consider adjusting reward function to provide more positive feedback")
    print("   - Check if transaction costs are too high relative to price movements")
if total_actions[1] / total_count > 0.7:
    print("   - Try reducing hold action penalty or increasing position change rewards")
    print("   - Consider adding momentum or trend-following features")
if enhanced_agent.epsilon < 0.1:
    print("   - Use slower epsilon decay for longer exploration phase")
    print("   - Consider epsilon scheduling with periodic increases")

print("\n" + "="*80)

## 10) Evaluate agent & simple backtest
Run the agent greedily over the same demo env to collect P&L and trades.

In [ ]:
# Greedy evaluation
state = env.reset()
done = False
equity = [0.0]
trade_log = []
step = 0
while not done and step < 500:
    action, info = agent.decide_action(state, mode='eval', minutes_to_close=180-step)
    next_state, reward, done, info = env.step(action)
    if reward != 0:
        trade_log.append({'step': step, 'reward': reward})
    equity.append(equity[-1] + reward)
    state = next_state
    step += 1
print('Evaluation complete. Trades:', len(trade_log))
import matplotlib.pyplot as plt
plt.plot(equity)
plt.title('Demo equity curve (synthetic demo)')
plt.xlabel('Step')
plt.ylabel('Cumulative reward')
plt.grid(True)
plt.show()

## 11) Save / load trained model weights
Save model weights and scaler for later inference.

In [ ]:
save_dir = repo_root / 'models'
save_dir.mkdir(exist_ok=True)
policy_path = save_dir / 'tf_policy_weights'
policy_model.save_weights(str(policy_path))
print('Saved policy weights to', policy_path)
# To load: policy_model.load_weights(str(policy_path))
import joblib
scaler_path = save_dir / 'feature_scaler.pkl'
joblib.dump(scaler, scaler_path)
print('Saved scaler to', scaler_path)

## 12) Visualize results (equity curve, trades)
We plotted a simple equity curve above. For real data, align entry/exit timestamps and mark trades on a price chart.

## 13) Quick smoke tests / assertions
A few checks to ensure shapes and GPU visibility.

In [ ]:
# Smoke assertions
s = env.reset()
assert s.shape == (30, len(feature_cols)), f'Unexpected state shape: {s.shape}'
q = policy_model(s[np.newaxis].astype(np.float32))
assert q.shape[-1] == 3, 'Policy output should have 3 actions'
print('Smoke checks passed. GPU available:', len(tf.config.list_physical_devices('GPU'))>0)

---
That completes the demo notebook for entry/exit policy in TensorFlow.
Next steps (suggested):
- Replace the `DemoEnv` with the repo environment (`IntradayTradingEnv`) and adapt the feature columns to match the state builder (lookback=30, feature_count=32 or 68).
- Extend the training loop per `models/training.py` to run over many episodes, add TensorBoard logging, and checkpointing.
- If you'd like, I can: (a) wire this notebook to the real env/data in the repo, or (b) convert the training loop into a script matching `training.py`. Which do you prefer?